This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data.

In [1]:
import os
import sys
from pathlib import Path
import pickle
import pyarrow

# Data processing and analysis
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    #!pip install git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root
    import corrosion_scoring as cs

Running in local (VSCode) environment


In [2]:
# environment check
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    # For Kaggle # Whole filtered Data
    eccontri_path = Path("/kaggle/input/eccontri-uniprot-enriched/ECcontri_Uniprot_enriched.parquet")
    ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)
    # Directory to output large files 
    large_dir =  Path("/kaggle/working/")
    # Directory to output large files # eccontris, compilated db

else:              
    # For Vscode 
    # large galaxies input and output #large size dir for large files hosted instead in kaggle
    large_dir = Path("/home/beatriz/MIC")
    # Directory to output large files
    output_large = large_dir / "output_large"
    # Whole filtered Data
    eccontri_path = output_large / 'ECcontri_Uniprot_enriched.parquet'


In [3]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

In [ ]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = [
            'idx', 'Genus', 'protein_name', 'EC', 'enzyme_names',
            'enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'corrosion_relevance', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]

    for d in global_terms_list:
        for category, terms in d.items():
            if isinstance(terms, dict):
                # Handle functional_categories special case, nested with scores
                if 'terms' in terms and 'score' in terms:
                    # This is functional_categories format: {'terms': [...], 'score': 1.5}
                    existing = []
                    for term in terms['terms']:  # Access the 'terms' key
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[category] = existing
                else:
                    # Handle other nested dictionaries
                    for subcategory, subterms in terms.items():
                        if isinstance(subterms, list):  # Making sure it's a list
                            existing = []
                            for term in subterms:
                                for col in cols_terms:
                                    if col in df.columns:
                                        if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                            existing.append(term)
                                            break
                            if existing:
                                found[f"{category}.{subcategory}"] = existing
            else:
                # Handle simple lists
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    
    return found
#sample= ECcontri_Uniprot_enriched.sample(n=15000)

In [ ]:
real_terms = validate_terms(ECcontri_Uniprot_enriched, [
    cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups
])

In [6]:
# Saving the new dataframe
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    rt_path = large_dir / 'real_terms.pkl'
else:
    rt_path = output_large / 'real_terms.pkl'

with open(rt_path, 'wb') as f:
    pickle.dump(real_terms, f)

In [7]:
# Reading the dictionaries
with open(rt_path, 'rb') as f:
    real_terms= pickle.load(f)

In [8]:
real_terms

{'iron': ['iron', 'heme', 'iron-sulfur', 'siderophore', 'ferritin'],
 'manganese': ['manganese', 'mn'],
 'copper': ['copper', 'cupric'],
 'nickel': ['Ni2+'],
 'cobalt': ['cobalt', 'cobalamin', 'vitamin B12'],
 'magnesium': ['magnesium'],
 'calcium': ['Ca2+', 'calcium'],
 'Mo': ['Mo', 'molybdenum', 'molybdopterin', 'molybdenum cofactor'],
 'V5+': ['V5+', 'vanadium'],
 'Al3+': ['Al3+'],
 'Cr3+': ['Cr3+', 'chromate'],
 'zinc': ['Zn2+', 'zinc'],
 'potassium': ['potassium'],
 'selenium': ['selenium', 'Se', 'selenocysteine', 'selenoprotein'],
 'phosphate': ['phosphate'],
 'nitrate': ['NO3-', 'nitrate'],
 'nitrite': ['nitrite'],
 'chloride': ['Cl-', 'chloride'],
 'sulfate': ['sulfate'],
 'sulfide': ['sulfide', 'desulfovibrio', 'h2s'],
 'thiosulfate': ['thiosulfate'],
 'oxygen': ['O2', 'oxygen', 'oxidase'],
 'hydrogen': ['hydrogenase', 'h2'],
 'organics': ['methane',
  'methane',
  'methanogenesis',
  'formate',
  'formate',
  'formic acid',
  'acetate',
  'acetate',
  'acetic acid',
  'propio